# 数据源回测对比：Tushare vs 掘金（Goldminer）日线数据

本 Notebook 在保持策略参数和交易逻辑完全不变的前提下，对比两个数据源的日线数据对回测结果的影响。

- Tushare 日线数据：`ts_stock_all_data`
- 掘金（Goldminer）日线数据：`gm_stock_all_data`

## 对比检查项

1. 两个数据源的共同日期覆盖范围
2. 数据列名规范化、单位统一（信号生成前预处理）
3. 信号重叠度与各自独有信号差异
4. 回测绩效对比与逐笔交易利润差异

## 关键处理说明

本地 Tushare 的 `total_mv/float_mv` 字段已存储为**亿元**单位，而掘金的 `total_mv/mv_A_free_float` 是原始**元**单位，因此需要除以 `1e8` 统一为亿元。这样才能保证 `mv_min=35, mv_max=1000` 的流通市值过滤条件对两个数据源是公平的。


In [16]:
# ============================================================
# 环境初始化：自动重载模块、导入依赖库与核心函数
# ============================================================
%reload_ext autoreload
%autoreload 2

import datetime as dt
from pathlib import Path

import numpy as np
import pandas as pd
import polars as pl

# 导入项目核心函数
from my_utils.fun import (
    DATA_ROOT_DIR,
    add_sma,
    cal_limit_avg_turnover,
    cal_n_lowest,
    get_logger,
    mark_last_limit_desc,
    mark_limit_desc,
    mark_limit_status,
    read_day_data,
)
from my_utils.trade_fun import cal_trade_info, trade

# 设置日志（输出到 log/ 目录下）
logging = get_logger(log_file='log/data_source_backtest_compare.log', inherit=False)

# Polars 显示设置（行列不截断，方便查看）
pl.Config.set_tbl_rows(20)
pl.Config.set_tbl_cols(40)

polars.config.Config

In [17]:
# ============================================================
# 策略参数配置（与回测 demo 保持一致）
# ============================================================

# 数据时间范围
REQUEST_START_DATE = dt.date(2025, 1, 1)
REQUEST_END_DATE = dt.date.today()

# 待对比的数据源映射：名称 -> 本地 Parquet 文件路径
DATA_SOURCES = {
    'ts': 'ts_stock_all_data',
    'gm': 'gm_stock_all_data',
}

# 策略筛选参数
PARAMS_DICT = {
    'low': -5,                        # 开盘涨幅下限（%）
    'high': -2.5,                     # 开盘涨幅上限（%）
    'mv_min': 35,                     # 最小流通市值（亿元）
    'mv_max': 1000,                   # 最大流通市值（亿元）
    'prev_limit_status': ['断板', '炸板'],  # 前一日涨停状态筛选
    'avg_limit_turnover_5_min': -1,   # 五分钟均换手率（负值表示不使用）
}

# 交易参数
BUY_DELAY_DAYS = 0          # 买入延迟天数
EXTEND_HOLDING_DAYS = 0     # 延长持仓天数
NEED_ADJ = True             # 是否使用复权价格
FEE_RATE = 0.004            # 单边手续费率（含印花税等）
STOP_LOSS_PCT = 0.09        # 止损阈值（-9%）
POSITION_WEIGHT = 0.4       # 每日总仓位。profit 是单票收益率，weight_profit = profit * 0.4 折算为账户收益贡献

In [18]:
# ============================================================
# 获取两个数据源的日期覆盖范围，确定共同对比区间
# ============================================================

def source_date_range(file_path: str) -> tuple[dt.date, dt.date]:
    """
    读取本地 Parquet 数据，获取最小和最大交易日日期

    Args:
        file_path: 数据源的文件路径（相对于 DATA_ROOT_DIR）

    Returns:
        (最小日期, 最大日期) 的元组
    """
    root = Path(DATA_ROOT_DIR) / file_path
    stat = (
        pl.scan_parquet(str(root))
        .select(
            pl.col('trading_date').min().alias('min_date'),
            pl.col('trading_date').max().alias('max_date'),
        )
        .collect()
    )
    return stat['min_date'][0], stat['max_date'][0]

# 计算每个数据源的实际日期范围
ranges = {name: source_date_range(path) for name, path in DATA_SOURCES.items()}

# 取两个数据源的交集，且不超出请求范围
common_start = max(REQUEST_START_DATE, *(r[0] for r in ranges.values()))
common_end = min(REQUEST_END_DATE, *(r[1] for r in ranges.values()))

print('数据源日期范围:', ranges)
print('共同对比区间:', common_start, '至', common_end)

if common_start > common_end:
    raise ValueError('两个数据源没有重叠的日期区间，无法对比')

数据源日期范围: {'ts': (datetime.date(2021, 1, 4), datetime.date(2026, 4, 7)), 'gm': (datetime.date(2021, 1, 4), datetime.date(2026, 4, 28))}
共同对比区间: 2025-01-01 至 2026-04-07


In [19]:
# ============================================================
# 加载原始日线数据并统一字段名和单位
# ============================================================

def load_and_normalize_daily_data(file_path: str) -> pl.DataFrame:
    """
    加载日线原始数据，统一字段名、单位、涨跌幅计算方式

    关键处理：
    - 掘金数据：市值字段是原始"元"单位，转换为"亿元"
    - Tushare 数据：映射 float_mv -> mv_A_free_float、turn_over -> turnover_rate
    - 统一涨跌幅为复权涨跌幅，不管原始数据的 pct 定义

    Args:
        file_path: 数据源 Parquet 文件路径

    Returns:
        规范化后的 DataFrame
    """
    df = read_day_data(start_date=common_start, end_date=common_end, file_path=file_path)

    if file_path == 'gm_stock_all_data':
        # 掘金：市值原始单位是元，转成亿元，与 Tushare 本地数据保持一致
        df = df.with_columns([
            (pl.col('mv_A_free_float') / 1e8).alias('mv_A_free_float'),
            (pl.col('total_mv') / 1e8).alias('total_mv'),
        ])
    elif file_path == 'ts_stock_all_data':
        # Tushare：字段名映射为标准名称，与策略统一接口
        df = df.with_columns([
            pl.col('float_mv').alias('mv_A_free_float'),
            pl.col('turn_over').alias('turnover_rate'),
            (pl.col('type') == 'ST').fill_null(False).alias('is_st'),
        ])
    else:
        raise ValueError(f'未知数据源: {file_path}')

    # 统一使用复权收盘价计算涨跌幅，避免不同数据源 pct 定义不一致
    df = df.with_columns(((pl.col('close') / pl.col('pre_close')) - 1).alias('pct'))
    return df

In [20]:
# ============================================================
# 特征工程 + 信号生成（涨停低开策略的核心逻辑）
# ============================================================

def build_signal_file(file_path: str) -> tuple[pl.DataFrame, pl.DataFrame]:
    """
    对指定数据源执行完整的特征工程，生成买入信号

    信号条件概览：
    - 非 ST、非创业板/科创板/北交所
    - 前一日状态为"断板"或"炸板"
    - 开盘涨幅在 [-5%, -2.5%] 之间
    - 前一日收盘价在 7日均线之上
    - 涨停描述不是"1天1板"
    - 流通市值在 [35亿, 1000亿] 之间

    Args:
        file_path: 数据源 Parquet 文件路径

    Returns:
        (stock_data, signal_file) 元组：
            stock_data: 全量数据（含计算字段）
            signal_file: 信号过滤后的数据（signal == 1）
    """
    stock_data = load_and_normalize_daily_data(file_path)

    # ---- 特征计算 ----
    stock_data = mark_limit_status(stock_data)          # 标记当日涨停状态
    stock_data = mark_limit_desc(stock_data)            # 标记几天几板
    stock_data = mark_last_limit_desc(stock_data)       # 标记最近一次涨停描述
    stock_data = cal_limit_avg_turnover(stock_data, window=5, turnover_col='turnover_rate')  # 5日均换手率

    stock_data = add_sma(stock_data, window=5)          # 5日均线
    stock_data = add_sma(stock_data, window=7)          # 7日均线

    # ---- 衍生字段 ----
    stock_data = stock_data.with_columns(
        ((pl.col('open') - pl.col('pre_close')) / pl.col('pre_close') * 100).alias('open_pct'),      # 开盘涨幅%
        ((pl.col('close') - pl.col('sma_7')) / pl.col('sma_7') * 100).alias('close_sma7_pct'),       # 收盘价偏离7均线%
        (pl.col('amount') * 100 / pl.col('volume')).alias('vwap'),                                    # 均价
        (pl.col('low') <= pl.col('limit_down') * 1.01).alias('touch_limit_down'),                     # 是否触碰跌停
    )
    stock_data = cal_n_lowest(stock_data)               # 计算 N 日最低价
    stock_data = stock_data.sort(['code', 'trading_date'])

    # ---- 前移字段（用前一日数据判断当日信号） ----
    stock_data = stock_data.with_columns([
        pl.col('limit_status').shift(1).over('code').alias('prev_limit_status'),
        pl.col('sma_7').shift(1).over('code').alias('prev_sma_7'),
        pl.col('pct').shift(1).over('code').alias('pre_pct'),
        pl.col('vwap').shift(1).over('code').alias('pre_vwap'),
        pl.col('close_sma7_pct').shift(1).over('code').alias('pre_close_sma7_pct'),
    ])

    # ---- 信号生成 ----
    stock_data = stock_data.with_columns(
        signal=pl.when(
            (~pl.col('is_st'))                                              # 非 ST
            & ~(pl.col('code').str.split('.').list[1].str.starts_with('30')  # 非创业板
                | pl.col('code').str.split('.').list[1].str.starts_with('688')  # 非科创板
                | pl.col('code').str.split('.').list[1].str.starts_with('90')   # 非北交所
                | pl.col('code').str.split('.').list[1].str.starts_with('20')   # 非创业板
            )
            & pl.col('prev_limit_status').is_in(PARAMS_DICT['prev_limit_status'])  # 前一日断板/炸板
            & (pl.col('open_pct') >= PARAMS_DICT['low'])                            # 开盘不低于 -5%
            & (pl.col('open_pct') <= PARAMS_DICT['high'])                           # 开盘不高于 -2.5%
            & (pl.col('pre_close') >= pl.col('prev_sma_7'))                         # 前收盘在7均线上
            & (pl.col('last_limit_desc') != '1天1板')                                # 排除 1天1板
            & pl.col('last_limit_desc').is_not_null()                                # 必须有涨停描述
            & (pl.col('mv_A_free_float') >= PARAMS_DICT['mv_min'])                  # 流通市值下限
            & (pl.col('mv_A_free_float') <= PARAMS_DICT['mv_max'])                  # 流通市值上限
            & ((pl.col('open') / pl.col('lowest_30')) <= 3)                         # 开盘价不超过30日最低的3倍
        ).then(1).otherwise(0)
    )

    signal_file = stock_data.filter(pl.col('signal') == 1)
    return stock_data, signal_file

In [21]:
# ============================================================
# 数据源基础质量检查：行数、股票数、空值、关键字段分布
# ============================================================

source_quality = []
for source_name, file_path in DATA_SOURCES.items():
    df = load_and_normalize_daily_data(file_path)
    source_quality.append(
        df.select(
            pl.lit(source_name).alias('source'),            # 数据源名称
            pl.len().alias('rows'),                          # 总行数
            pl.col('code').n_unique().alias('n_codes'),      # 股票数量
            pl.col('trading_date').min().alias('min_date'),  # 最小日期
            pl.col('trading_date').max().alias('max_date'),  # 最大日期
            pl.col('open').null_count().alias('open_nulls'),             # 开盘价空值
            pl.col('pre_close').null_count().alias('pre_close_nulls'),   # 前收盘价空值
            pl.col('limit_up').null_count().alias('limit_up_nulls'),      # 涨停价空值
            pl.col('limit_down').null_count().alias('limit_down_nulls'),  # 跌停价空值
            pl.col('mv_A_free_float').median().alias('free_mv_median_100m_cny'),  # 流通市值中位数(亿)
            pl.col('turnover_rate').median().alias('turnover_median'),    # 换手率中位数
            pl.col('is_st').sum().alias('st_rows'),          # ST 行数
        )
    )
source_quality_df = pl.concat(source_quality)
source_quality_df

source,rows,n_codes,min_date,max_date,open_nulls,pre_close_nulls,limit_up_nulls,limit_down_nulls,free_mv_median_100m_cny,turnover_median,st_rows
str,u32,u32,date,date,u32,u32,u32,u32,f64,f64,u32
"""ts""",1560379,5236,2025-01-02,2026-04-07,0,0,478,478,54.32,2.27,49236
"""gm""",1583045,5309,2025-01-02,2026-04-07,6,0,0,0,54.553411,2.2284,49346


In [22]:
# ============================================================
# 分别生成两个数据源的信号，对比每日信号数量差异
# ============================================================

built = {}
signal_summary = []
for source_name, file_path in DATA_SOURCES.items():
    stock_data, signal_file = build_signal_file(file_path)
    built[source_name] = {'stock_data': stock_data, 'signal_file': signal_file}
    signal_summary.append(signal_file.group_by('trading_date').agg(pl.len().alias(f'{source_name}_signals')))
    print(source_name, file_path, '信号数量:', signal_file.height)

# 按交易日对齐，比较每日信号数
signal_summary_df = signal_summary[0].join(signal_summary[1], on='trading_date', how='full', coalesce=True).sort('trading_date')
signal_summary_df = signal_summary_df.with_columns([
    pl.col('ts_signals').fill_null(0),
    pl.col('gm_signals').fill_null(0),
    (pl.col('gm_signals').fill_null(0) - pl.col('ts_signals').fill_null(0)).alias('gm_minus_ts'),  # 掘金信号数 - Tushare信号数
])
signal_summary_df

ts ts_stock_all_data 信号数量: 1097
gm gm_stock_all_data 信号数量: 1112


trading_date,ts_signals,gm_signals,gm_minus_ts
date,u32,u32,u32
2025-01-07,2,2,0
2025-01-08,4,4,0
2025-01-09,8,8,0
2025-01-10,2,2,0
2025-01-13,5,5,0
2025-01-15,2,2,0
2025-01-16,5,5,0
2025-01-17,3,3,0
2025-01-20,3,3,0


In [23]:
# ============================================================
# 信号重叠分析：两个数据源生成的信号在(日期, 股票)维度上的重叠情况
# ============================================================

def signal_key_df(signal_file: pl.DataFrame, source_name: str) -> pl.DataFrame:
    """提取 (交易日, 股票代码) 主键，标记信号属于哪个数据源"""
    return signal_file.select(['trading_date', 'code']).with_columns(pl.lit(1).alias(f'in_{source_name}'))

ts_keys = signal_key_df(built['ts']['signal_file'], 'ts')
gm_keys = signal_key_df(built['gm']['signal_file'], 'gm')

signal_diff = (
    ts_keys.join(gm_keys, on=['trading_date', 'code'], how='full', coalesce=True)
    .with_columns([pl.col('in_ts').fill_null(0), pl.col('in_gm').fill_null(0)])
    .with_columns(
        pl.when((pl.col('in_ts') == 1) & (pl.col('in_gm') == 1)).then(pl.lit('both'))         # 两数据源都有信号
        .when(pl.col('in_ts') == 1).then(pl.lit('ts_only'))                                     # 仅 Tushare 有信号
        .otherwise(pl.lit('gm_only'))                                                            # 仅掘金有信号
        .alias('signal_group')
    )
)

signal_diff.group_by('signal_group').len().sort('signal_group')

signal_group,len
str,u32
"""both""",1097
"""gm_only""",15


In [24]:
# ============================================================
# 字段级差异分析：对比同一(日期, 股票)在 Tushare 和掘金中的价格、市值等
# ============================================================

compare_cols = ['trading_date', 'code', 'open', 'pre_close', 'limit_up', 'limit_down', 'low', 'close', 'mv_A_free_float', 'turnover_rate', 'is_st']

# 分别提取两个数据源的字段，用 _ts / _gm 后缀区分
ts_daily = built['ts']['stock_data'].select(compare_cols).rename({c: f'{c}_ts' for c in compare_cols if c not in ['trading_date', 'code']})
gm_daily = built['gm']['stock_data'].select(compare_cols).rename({c: f'{c}_gm' for c in compare_cols if c not in ['trading_date', 'code']})

daily_field_compare = ts_daily.join(gm_daily, on=['trading_date', 'code'], how='inner')

# 计算每个字段的差值（掘金 - Tushare）
for c in ['open', 'pre_close', 'limit_up', 'limit_down', 'low', 'close', 'mv_A_free_float', 'turnover_rate']:
    daily_field_compare = daily_field_compare.with_columns((pl.col(f'{c}_gm') - pl.col(f'{c}_ts')).alias(f'{c}_diff'))

# 汇总：各字段绝对差均值、ST 标记不匹配的行数
field_diff_summary = daily_field_compare.select(
    pl.len().alias('matched_rows'),       # 匹配总行数
    *[pl.col(f'{c}_diff').abs().mean().alias(f'{c}_abs_mean_diff') for c in ['open', 'pre_close', 'limit_up', 'limit_down', 'low', 'close', 'mv_A_free_float', 'turnover_rate']],
    (pl.col('is_st_ts') != pl.col('is_st_gm')).sum().alias('st_flag_mismatch_rows'),  # ST标记不一致的行数
)
field_diff_summary

matched_rows,open_abs_mean_diff,pre_close_abs_mean_diff,limit_up_abs_mean_diff,limit_down_abs_mean_diff,low_abs_mean_diff,close_abs_mean_diff,mv_A_free_float_abs_mean_diff,turnover_rate_abs_mean_diff,st_flag_mismatch_rows
u32,f64,f64,f64,f64,f64,f64,f64,f64,u32
1558686,0.0,0.000181,192.855017,0.00019,0.0,0.0,0.159336,0.006748,0


In [25]:
# ============================================================
# 分别对两个数据源的信号执行回测，使用完全相同的交易逻辑
# ============================================================

def run_backtest_for_source(source_name: str, file_path: str) -> dict:
    """
    对指定数据源的信号文件执行完整回测

    Args:
        source_name: 数据源名称（用于标记）
        file_path: 数据源 Parquet 文件路径（用于回测时读取日线数据）

    Returns:
        dict: {'result_df': 汇总指标, 'merged_df': 逐笔交易明细}
    """
    signal_file = built[source_name]['signal_file']
    logging.info(f'开始回测 {source_name}: {file_path}, 信号数={signal_file.height}')
    result_df, merged_df = cal_trade_info(
        signal_file,
        trade_fun=trade,
        start_date=common_start.strftime('%Y-%m-%d'),
        end_date=common_end.strftime('%Y-%m-%d'),
        buy_delay_days=BUY_DELAY_DAYS,
        extend_holding_days=EXTEND_HOLDING_DAYS,
        day_data_file_path=file_path,
    )
    if isinstance(merged_df, pl.DataFrame):
        merged_df = merged_df.with_columns([
            pl.lit(source_name).alias('source'),                                                     # 标记数据源
            (pl.col('profit') * POSITION_WEIGHT).alias('weight_profit'),                             # 单票收益 * 总仓位 = 账户收益贡献
        ])
    return {'result_df': result_df, 'merged_df': merged_df}

# 分别回测并保存结果
backtests = {source_name: run_backtest_for_source(source_name, file_path) for source_name, file_path in DATA_SOURCES.items()}

开始回测 ts: ts_stock_all_data, 信号数=1097
使用40个进程并行处理，共283个日期任务
回测任务进度: 100%|███████████████████████| 283/283 [00:11<00:00, 24.66个日期/s]
所有回测任务完成，共处理283个日期的结果
开始回测 gm: gm_stock_all_data, 信号数=1112
使用40个进程并行处理，共283个日期任务
回测任务进度: 100%|███████████████████████| 283/283 [00:08<00:00, 34.99个日期/s]
所有回测任务完成，共处理283个日期的结果


In [26]:
# ============================================================
# 绩效指标计算与对比（胜率、盈亏比、累计收益等）
# ============================================================

def calc_perf(merged_df: pl.DataFrame, profit_col: str = 'weight_profit') -> dict:
    """
    从逐笔交易明细中计算核心绩效指标

    注意：默认使用 weight_profit（单票收益 * 总仓位 0.4）作为收益列，
    因为 calc_perf 中对同一卖出日期的多笔交易取均值，相当于
    每日账户收益 = avg(当日各票收益) * 总仓位

    Args:
        merged_df: 逐笔交易明细（含 profit, sell_time, holding_days 等字段）
        profit_col: 用于计算的收益列名（默认 weight_profit）

    Returns:
        dict: 包含交易次数、胜率、盈亏比、累计收益等指标
    """
    df = merged_df.filter(pl.col(profit_col).is_not_null())
    if df.height == 0:
        return {'trades': 0, 'sell_days': 0, 'total_return_compound': np.nan, 'total_return_simple': np.nan, 'win_rate': np.nan, 'avg_profit': np.nan, 'avg_win': np.nan, 'avg_loss': np.nan, 'profit_loss_ratio': np.nan, 'avg_holding_days': np.nan}

    pdf = df.select(['sell_time', profit_col, 'profit', 'holding_days']).to_pandas()
    pdf['sell_date'] = pd.to_datetime(pdf['sell_time']).dt.date

    # 按卖出日期分组，对同一日所有票的 weight_profit 取均值 = 当日账户收益率
    daily_ret = pdf.groupby('sell_date')[profit_col].mean() / 100

    # 复合累计收益（考虑复利效应）
    total_compound = (1 + daily_ret).prod() - 1

    # 胜率：剔除收益为 0 的交易
    non_zero = pdf[pdf['profit'] != 0]
    win_rate = (non_zero['profit'] > 0).mean() if len(non_zero) else np.nan

    avg_win = pdf.loc[pdf['profit'] > 0, 'profit'].mean()
    avg_loss = abs(pdf.loc[pdf['profit'] < 0, 'profit'].mean())

    return {
        'trades': len(pdf),                      # 总交易次数
        'sell_days': daily_ret.shape[0],          # 卖出交易日数
        'total_return_compound': total_compound,  # 复合累计收益率
        'total_return_simple': daily_ret.sum(),   # 简单加总收益率
        'win_rate': win_rate,                     # 胜率
        'avg_profit': pdf['profit'].mean(),       # 单笔平均收益(%)
        'avg_win': avg_win,                       # 盈利交易平均收益(%)
        'avg_loss': avg_loss,                     # 亏损交易平均亏损(%)
        'profit_loss_ratio': avg_win / avg_loss if avg_loss and not np.isnan(avg_loss) else np.nan,  # 盈亏比
        'avg_holding_days': pdf['holding_days'].dropna().mean(),  # 平均持仓天数
    }

perf_df = pd.DataFrame([{'source': source_name, **calc_perf(obj['merged_df'])} for source_name, obj in backtests.items()])
perf_df

,source,trades,sell_days,total_return_compound,total_return_simple,win_rate,avg_profit,avg_win,avg_loss,profit_loss_ratio,avg_holding_days
0,ts,1097,278,3.058759,1.504430,0.474932,1.047089,8.808085,5.972839,1.474690,1.659982
1,gm,1112,279,3.210130,1.538995,0.476619,1.073817,8.821004,5.981182,1.474793,1.665917


In [27]:
# ============================================================
# 逐笔交易对比：按(信号日期, 股票代码)匹配后，对比买卖价格和收益
# ============================================================

trade_cols = ['trading_date', 'code', 'buy_time', 'buy_price', 'sell_time', 'sell_price', 'profit', 'holding_days', 'sell_reason']

# 分别提取交易明细，用 _ts / _gm 后缀区分
ts_trades = backtests['ts']['merged_df'].select([c for c in trade_cols if c in backtests['ts']['merged_df'].columns]).rename({c: f'{c}_ts' for c in trade_cols if c not in ['trading_date', 'code']})
gm_trades = backtests['gm']['merged_df'].select([c for c in trade_cols if c in backtests['gm']['merged_df'].columns]).rename({c: f'{c}_gm' for c in trade_cols if c not in ['trading_date', 'code']})

# 按 (交易日, 股票) 全外连接，保留双方单独有的交易
trade_compare = ts_trades.join(gm_trades, on=['trading_date', 'code'], how='full', coalesce=True)
trade_compare = trade_compare.with_columns([
    (pl.col('profit_gm') - pl.col('profit_ts')).alias('profit_diff_gm_minus_ts'),  # 收益差异(掘金 - Tushare)
    (pl.col('buy_price_gm') - pl.col('buy_price_ts')).alias('buy_price_diff'),      # 买入价差异
    (pl.col('sell_price_gm') - pl.col('sell_price_ts')).alias('sell_price_diff'),   # 卖出价差异
])

# 汇总统计
trade_compare_summary = trade_compare.select(
    pl.len().alias('rows'),                                                              # 总行数
    pl.col('profit_ts').is_not_null().sum().alias('ts_trade_rows'),                      # Tushare 交易数
    pl.col('profit_gm').is_not_null().sum().alias('gm_trade_rows'),                      # 掘金交易数
    (pl.col('profit_ts').is_not_null() & pl.col('profit_gm').is_not_null()).sum().alias('matched_trade_rows'),  # 重叠交易数
    pl.col('profit_diff_gm_minus_ts').abs().mean().alias('abs_profit_diff_mean'),         # 收益绝对差均值
    pl.col('profit_diff_gm_minus_ts').mean().alias('profit_diff_mean'),                   # 收益差均值
    pl.col('buy_price_diff').abs().mean().alias('abs_buy_price_diff_mean'),               # 买入价绝对差均值
    pl.col('sell_price_diff').abs().mean().alias('abs_sell_price_diff_mean'),             # 卖出价绝对差均值
)
trade_compare_summary

rows,ts_trade_rows,gm_trade_rows,matched_trade_rows,abs_profit_diff_mean,profit_diff_mean,abs_buy_price_diff_mean,abs_sell_price_diff_mean
u32,u32,u32,u32,f64,f64,f64,f64
1112,1097,1112,1097,0.013756,-0.013756,0.0,0.001632


In [28]:
# ============================================================
# 查看收益差异最大的前50笔交易（帮助定位数据源差异根源）
# ============================================================

trade_compare.sort(pl.col('profit_diff_gm_minus_ts').abs(), descending=True).head(50)

trading_date,code,buy_time_ts,buy_price_ts,sell_time_ts,sell_price_ts,profit_ts,holding_days_ts,sell_reason_ts,buy_time_gm,buy_price_gm,sell_time_gm,sell_price_gm,profit_gm,holding_days_gm,sell_reason_gm,profit_diff_gm_minus_ts,buy_price_diff,sell_price_diff
date,str,datetime[μs],f64,datetime[μs],f64,f64,f64,str,datetime[μs],f64,datetime[μs],f64,f64,f64,str,f64,f64,f64
2025-07-24,"""SHSE.600930""",null,null,null,null,null,null,null,2025-07-24 09:30:00,7.35,2025-07-25 11:30:00,6.83,-7.81527,1.5,"""未涨停卖出""",null,null,null
2025-11-27,"""SHSE.603216""",null,null,null,null,null,null,null,2025-11-27 09:30:00,23.01,2025-12-01 11:30:00,30.48,31.408655,2.5,"""未涨停卖出""",null,null,null
2025-11-13,"""SZSE.002387""",null,null,null,null,null,null,null,2025-11-13 09:30:00,9.8,2025-11-14 11:30:00,9.83,-0.49313,1.5,"""未涨停卖出""",null,null,null
2025-12-04,"""SZSE.000592""",null,null,null,null,null,null,null,2025-12-04 09:30:00,10.9,2025-12-05 11:30:00,11.71,6.575167,1.5,"""未涨停卖出""",null,null,null
2025-12-08,"""SZSE.002348""",null,null,null,null,null,null,null,2025-12-08 09:30:00,5.95,2025-12-09 11:30:00,6.65,10.87415,1.5,"""未涨停卖出""",null,null,null
2025-12-05,"""SZSE.000592""",null,null,null,null,null,null,null,2025-12-05 09:30:00,10.8,2025-12-08 11:30:00,12.27,12.705843,1.5,"""未涨停卖出""",null,null,null
2025-12-09,"""SZSE.002348""",null,null,null,null,null,null,null,2025-12-09 09:30:00,6.58,2025-12-10 11:30:00,6.23,-6.073578,1.5,"""未涨停卖出""",null,null,null
2026-01-13,"""SHSE.600337""",null,null,null,null,null,null,null,2026-01-13 09:30:00,3.62,2026-01-14 11:30:00,3.49,-4.359358,1.5,"""未涨停卖出""",null,null,null
2026-01-06,"""SHSE.603608""",null,null,null,null,null,null,null,2026-01-06 09:30:00,10.17,2026-01-07 11:30:00,9.68,-5.576514,1.5,"""未涨停卖出""",null,null,null


In [29]:
# ============================================================
# 查看仅在一个数据源中出现的独有信号
# ============================================================

signal_diff.filter(pl.col('signal_group') != 'both').sort(['trading_date', 'code']).head(100)

trading_date,code,in_ts,in_gm,signal_group
date,str,i32,i32,str
2025-07-24,"""SHSE.600930""",0,1,"""gm_only"""
2025-11-13,"""SZSE.002387""",0,1,"""gm_only"""
2025-11-27,"""SHSE.603216""",0,1,"""gm_only"""
2025-12-04,"""SZSE.000592""",0,1,"""gm_only"""
2025-12-05,"""SZSE.000592""",0,1,"""gm_only"""
2025-12-08,"""SZSE.002348""",0,1,"""gm_only"""
2025-12-09,"""SZSE.002348""",0,1,"""gm_only"""
2026-01-06,"""SHSE.603608""",0,1,"""gm_only"""
2026-01-12,"""SHSE.600337""",0,1,"""gm_only"""


In [30]:
# ============================================================
# （可选）导出对比结果到 CSV 文件
# ============================================================
# 如需导出，取消下方注释即可
# out_dir = Path('log')
# out_dir.mkdir(exist_ok=True)
# timestamp = dt.datetime.now().strftime('%Y%m%d_%H%M%S')
# perf_df.to_csv(out_dir / f'data_source_perf_compare_{timestamp}.csv', index=False, encoding='utf-8-sig')
# trade_compare.write_csv(out_dir / f'data_source_trade_compare_{timestamp}.csv', include_bom=True)
# signal_diff.write_csv(out_dir / f'data_source_signal_compare_{timestamp}.csv', include_bom=True)